# PRS-CS & PRS-CSx Evaluation — altPRS config

Fixed hyperparameters (no phi tuning here):
- **PRS-CS**: ϕ = auto, LDREF = EUR
- **PRS-CSx**: ϕ = auto, LDREF = META

Evaluation split:
- Train on GACRS train (60 % of full GACRS = 75 % of the upstream 80/20 train+test split with seed 42).
- Test  on GACRS test  (20 % of full GACRS).
- Test  on CAMP-only (full 597 cases + 64 controls) and CAMP-only Balanced (64v64 × 100 bootstrap).

External cohort evaluations (CAMP+1KG, CAMP+GTEx) have been removed.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns


def odds_ratio_quantile(y_true, y_pred, q=0.25):
    """Odds ratio: top quartile vs bottom quartile of predicted risk."""
    thresh_high = np.quantile(y_pred, 1 - q)
    thresh_low  = np.quantile(y_pred, q)
    top    = y_true[y_pred >= thresh_high]
    bottom = y_true[y_pred <= thresh_low]
    a = top.sum();    b = len(top) - a
    c = bottom.sum(); d = len(bottom) - c
    if b == 0 or c == 0:
        return np.inf
    return (a * d) / (b * c)


## 1. Load PRS-CS / PRS-CSx data + pheno splits

Only CAMP samples are kept from the mixed CAMP+1KG file (`Population == 'CAMP'`).
The CAMP+GTEx file is not used. Evaluation is on GACRS test + CAMP-only (full
+ balanced 64v64 bootstrap).


In [ ]:
# ============================================================
# INPUT_ROOT — where the input PRS melt CSVs + pheno splits live.
#   INPUT_ROOT/files/03_prscs-*_prscs{,x}_camp-1k1k-data-melt-admixture.csv
#   INPUT_ROOT/files/03_prscs-*_prscs{,x}-gacrs-only-data-melt-admixture.csv
#   INPUT_ROOT/combine/data/updated_pheno_gacrs_{val,train_test}.txt
#
# OUTPUT_ROOT — where per-sample predictions get saved.
#   OUTPUT_ROOT/prscs_evaluation/prs_predictions.csv
# ============================================================
from pathlib import Path
INPUT_ROOT  = Path('/Users/nancyh/Desktop/hartwell/gene_model/score')
OUTPUT_ROOT = Path('/Users/nancyh/Desktop/asthma-prs-study-fresh/09_ptrs-unified_model-evaluation/data/predictions')

files_dir  = INPUT_ROOT / 'files'
pheno_dir  = INPUT_ROOT / 'combine' / 'data'
ARTIFACT_DIR = OUTPUT_ROOT / 'prscs_evaluation'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

import_prefix = "03_prscs-prscsx-camp-gtex-onek1k-visualization"

# PRS-CS
prscs_gacrs = pd.read_csv(f"{files_dir}/{import_prefix}_prscs-gacrs-only-data-melt-admixture.csv")
prscs_camp  = pd.read_csv(f"{files_dir}/{import_prefix}_prscs_camp-1k1k-data-melt-admixture.csv")

# PRS-CSx
prscsx_gacrs = pd.read_csv(f"{files_dir}/{import_prefix}_prscsx_gacrs-only-data-melt-admixture.csv")
prscsx_camp  = pd.read_csv(f"{files_dir}/{import_prefix}_prscsx_camp-1k1k-data-melt-admixture.csv")

# Filter CAMP+1KG mix to CAMP-only samples immediately.
prscs_camp   = prscs_camp[prscs_camp['Population']  == 'CAMP'].copy()
prscsx_camp  = prscsx_camp[prscsx_camp['Population'] == 'CAMP'].copy()

# Pheno splits (upstream 80/20 GACRS split with seed 42 in 08_ptrs-construction).
pheno_val = pd.read_csv(pheno_dir / 'updated_pheno_gacrs_val.txt', sep=' ', header=0)
pheno_val.columns = ['v1', 'IID', 'V3', 'V4', 'V5', 'V6', 'IID_base', 'asthma']

pheno_train_test = pd.read_csv(pheno_dir / 'updated_pheno_gacrs_train_test.txt', sep=' ', header=0)
pheno_train_test.columns = ['v1', 'IID', 'V3', 'V4', 'V5', 'V6', 'IID_base', 'asthma']

val_ids        = set(pheno_val['IID'].values)
train_test_ids = set(pheno_train_test['IID'].values)

print(f"INPUT_ROOT   = {INPUT_ROOT}")
print(f"OUTPUT_ROOT  = {OUTPUT_ROOT}")
print()
print(f"PRS-CS  GACRS: {prscs_gacrs['IID'].nunique()}  CAMP-only: {prscs_camp['IID'].nunique()}")
print(f"PRS-CSx GACRS: {prscsx_gacrs['IID'].nunique()} CAMP-only: {prscsx_camp['IID'].nunique()}")
print(f"Val IDs (GACRS): {len(val_ids)}  Train+Test IDs (GACRS): {len(train_test_ids)}")


## 2. Assign GACRS train / val / test split labels

The upstream 80/20 GACRS split (seed 42, `08_ptrs-construction`) already produced a validation hold-out (`pheno_val`) and a train+test set (`pheno_train_test`). Here we further partition train+test into train (75 %) and test (25 %) with `random_state=0`, matching the split used in [meta_model_exploration_unified.ipynb](./meta_model_exploration_unified.ipynb).

In [ ]:
from sklearn.model_selection import train_test_split

# Assign split labels to PRS-CS GACRS data
prscs_gacrs['split'] = 'unknown'
prscs_gacrs.loc[prscs_gacrs['IID'].isin(val_ids), 'split'] = 'val'
prscs_gacrs.loc[prscs_gacrs['IID'].isin(train_test_ids), 'split'] = 'train_test'

# Same for PRS-CSx
prscsx_gacrs['split'] = 'unknown'
prscsx_gacrs.loc[prscsx_gacrs['IID'].isin(val_ids), 'split'] = 'val'
prscsx_gacrs.loc[prscsx_gacrs['IID'].isin(train_test_ids), 'split'] = 'train_test'

# Further split train_test into train (75%) and test (25%)
phi_col_cs = 'PRS-CS(ϕ)'
phi_col_csx = [c for c in prscsx_gacrs.columns if 'ϕ' in c][0]

tt_ids = prscs_gacrs[(prscs_gacrs['split'] == 'train_test') & (prscs_gacrs[phi_col_cs] == 'ϕ=auto')]['IID'].values
tt_pheno = prscs_gacrs[(prscs_gacrs['split'] == 'train_test') & (prscs_gacrs[phi_col_cs] == 'ϕ=auto')].set_index('IID')['PHENO']

train_ids_prs, test_ids_prs = train_test_split(
    tt_ids, test_size=0.25, stratify=tt_pheno.loc[tt_ids], random_state=0
)

# Apply to PRS-CS
prscs_gacrs.loc[(prscs_gacrs['IID'].isin(train_ids_prs)) & (prscs_gacrs['split'] == 'train_test'), 'split'] = 'train'
prscs_gacrs.loc[(prscs_gacrs['IID'].isin(test_ids_prs)) & (prscs_gacrs['split'] == 'train_test'), 'split'] = 'test'

# Apply to PRS-CSx
prscsx_gacrs.loc[(prscsx_gacrs['IID'].isin(train_ids_prs)) & (prscsx_gacrs['split'] == 'train_test'), 'split'] = 'train'
prscsx_gacrs.loc[(prscsx_gacrs['IID'].isin(test_ids_prs)) & (prscsx_gacrs['split'] == 'train_test'), 'split'] = 'test'

# Verify PRS-CS splits
print("=== PRS-CS splits ===")
for phi in prscs_gacrs[phi_col_cs].unique():
    subset = prscs_gacrs[prscs_gacrs[phi_col_cs] == phi]
    print(f"{phi}: val={( subset['split']=='val').sum()}, train={(subset['split']=='train').sum()}, test={(subset['split']=='test').sum()}")

# Verify PRS-CSx splits
print("\n=== PRS-CSx splits (per phi, META LDREF) ===")
for phi in prscsx_gacrs[phi_col_csx].unique():
    subset = prscsx_gacrs[(prscsx_gacrs[phi_col_csx] == phi) & (prscsx_gacrs['LDREF'] == 'META')]
    print(f"{phi}: val={(subset['split']=='val').sum()}, train={(subset['split']=='train').sum()}, test={(subset['split']=='test').sum()}")

## 3. altPRS configuration

Hard-code the PRS-CS(x) hyperparameters to the **altPRS** setting
(PRS-CS: ϕ=auto, LDREF=EUR; PRS-CSx: ϕ=auto, LDREF=META), which is what the
downstream integrated PRS + PTRS model uses. We do not perform validation-set
phi tuning here — the setting is fixed by the analysis choice, not selected
per this notebook.


In [ ]:
# altPRS: PRS-CS ϕ=auto/EUR, PRS-CSx ϕ=auto/META
best_prscs_phi    = 'ϕ=auto'
best_prscsx_phi   = 'ϕ=auto'
best_prscsx_ldref = 'META'

print(f"PRS-CS  hyperparams:  ϕ = {best_prscs_phi}  LDREF = EUR")
print(f"PRS-CSx hyperparams:  ϕ = {best_prscsx_phi}  LDREF = {best_prscsx_ldref}")


## 4. Train on GACRS train, evaluate on GACRS test + CAMP-only (full + balanced 64v64)


In [ ]:
# Evaluate both PRS-CS and PRS-CSx with the altPRS hyperparameters
from sklearn.utils import resample

n_repeats = 100
X_columns_prs  = ["PRS"]
X_columns_full = ["PRS", "1000G AFR", "1000G AMR", "1000G EAS", "1000G EUR", "1000G SAS"]

eval_configs = [
    ('PRS-CS',  'PRS only',           X_columns_prs),
    ('PRS-CS',  'PRS + Ancestry PCs', X_columns_full),
    ('PRS-CSx', 'PRS only',           X_columns_prs),
    ('PRS-CSx', 'PRS + Ancestry PCs', X_columns_full),
]

# ---- Assemble per-method data splits (GACRS train / test / val + CAMP-only) ----
data_sources = {}

# PRS-CS
data_sources['PRS-CS'] = {
    'train':     prscs_gacrs[(prscs_gacrs[phi_col_cs] == best_prscs_phi) & (prscs_gacrs['split'] == 'train')].set_index('IID'),
    'test':      prscs_gacrs[(prscs_gacrs[phi_col_cs] == best_prscs_phi) & (prscs_gacrs['split'] == 'test')].set_index('IID'),
    'val':       prscs_gacrs[(prscs_gacrs[phi_col_cs] == best_prscs_phi) & (prscs_gacrs['split'] == 'val')].set_index('IID'),
    'camp_only': prscs_camp[prscs_camp[phi_col_cs] == best_prscs_phi].set_index('IID'),
}

# PRS-CSx (META LDREF)
prscsx_best      = prscsx_gacrs[(prscsx_gacrs[phi_col_csx] == best_prscsx_phi) & (prscsx_gacrs['LDREF'] == best_prscsx_ldref)]
prscsx_camp_best = prscsx_camp[(prscsx_camp[phi_col_csx] == best_prscsx_phi) & (prscsx_camp['LDREF'] == best_prscsx_ldref)]

data_sources['PRS-CSx'] = {
    'train':     prscsx_best[prscsx_best['split'] == 'train'].set_index('IID'),
    'test':      prscsx_best[prscsx_best['split'] == 'test'].set_index('IID'),
    'val':       prscsx_best[prscsx_best['split'] == 'val'].set_index('IID'),
    'camp_only': prscsx_camp_best.set_index('IID'),
}

# ---- Evaluate each (method × config) on GACRS test + CAMP-only full + CAMP-only balanced ----
results = []

for method, config_name, x_cols in eval_configs:
    ds = data_sources[method]
    model = LogisticRegression(solver='liblinear', max_iter=1000)
    model.fit(ds['train'][x_cols], ds['train']['PHENO'])

    # --- GACRS test (single-pass) ---
    te      = ds['test']
    te_pred = model.predict_proba(te[x_cols])[:, 1]
    te_y    = te['PHENO'].values
    _, te_p = ttest_ind(te_pred[te_y == 0], te_pred[te_y == 1], equal_var=False)
    results.append({
        'Method': method, 'Config': config_name, 'Eval_Set': 'GACRS Test',
        'AUC': roc_auc_score(te_y, te_pred), 'AUC_std': np.nan,
        'AUPRC': average_precision_score(te_y, te_pred),
        'OR': odds_ratio_quantile(te_y, te_pred), 'OR_std': np.nan,
        'Mean_Diff': te_pred[te_y == 1].mean() - te_pred[te_y == 0].mean(),
        'P_Value': te_p,
        'N': len(te_y), 'N_cases': int(te_y.sum()), 'N_controls': int((te_y == 0).sum()),
    })

    # --- CAMP-only full (single-pass, unbalanced) ---
    co      = ds['camp_only']
    co_pred = model.predict_proba(co[x_cols])[:, 1]
    co_y    = co['PHENO'].values
    _, co_p = ttest_ind(co_pred[co_y == 0], co_pred[co_y == 1], equal_var=False)
    results.append({
        'Method': method, 'Config': config_name, 'Eval_Set': 'CAMP-only Full',
        'AUC': roc_auc_score(co_y, co_pred), 'AUC_std': np.nan,
        'AUPRC': average_precision_score(co_y, co_pred),
        'OR': odds_ratio_quantile(co_y, co_pred), 'OR_std': np.nan,
        'Mean_Diff': co_pred[co_y == 1].mean() - co_pred[co_y == 0].mean(),
        'P_Value': co_p,
        'N': len(co_y), 'N_cases': int(co_y.sum()), 'N_controls': int((co_y == 0).sum()),
    })

    # --- CAMP-only Balanced 64v64 × 100 bootstrap ---
    camp_cases    = co[co['PHENO'] == 1]
    camp_controls = co[co['PHENO'] == 0]
    n_ctrl = len(camp_controls)
    aucs, ors, diffs, auprcs = [], [], [], []
    for seed in range(n_repeats):
        cs = resample(camp_cases, n_samples=n_ctrl, random_state=seed, replace=False)
        bal = pd.concat([cs, camp_controls])
        y_bal = bal['PHENO'].values
        p_bal = model.predict_proba(bal[x_cols])[:, 1]
        aucs.append(roc_auc_score(y_bal, p_bal))
        auprcs.append(average_precision_score(y_bal, p_bal))
        ors.append(odds_ratio_quantile(y_bal, p_bal))
        diffs.append(p_bal[y_bal == 1].mean() - p_bal[y_bal == 0].mean())
    results.append({
        'Method': method, 'Config': config_name,
        'Eval_Set': f'CAMP-only Balanced ({n_ctrl}v{n_ctrl})',
        'AUC': np.mean(aucs), 'AUC_std': np.std(aucs),
        'AUPRC': np.mean(auprcs),
        'OR': np.mean(ors), 'OR_std': np.std(ors),
        'Mean_Diff': np.mean(diffs), 'P_Value': np.nan,
        'N': n_ctrl * 2, 'N_cases': n_ctrl, 'N_controls': n_ctrl,
    })

results_df = pd.DataFrame(results)

# Save the full result table
out_results = ARTIFACT_DIR / 'prscs_evaluation_results.csv'
results_df.to_csv(out_results, index=False)
print(f"Saved -> {out_results}")
print("\n=== PRS-CS & PRS-CSx Evaluation (trained on GACRS train) ===")
results_df


### Save per-sample predictions


In [ ]:
# === Save per-sample PRS-CS / PRS-CSx predictions ===
eval_sets_to_save = {
    'train':     'GACRS Train',
    'val':       'GACRS Val',
    'test':      'GACRS Test',
    'camp_only': 'CAMP-only',
}

prs_pred_rows = []
for method, config_name, x_cols in eval_configs:
    ds = data_sources[method]
    model = LogisticRegression(solver='liblinear', max_iter=1000)
    model.fit(ds['train'][x_cols], ds['train']['PHENO'])
    for key, label in eval_sets_to_save.items():
        eval_data = ds[key]
        if len(eval_data) == 0:
            continue
        preds  = model.predict_proba(eval_data[x_cols])[:, 1]
        y_true = eval_data['PHENO'].values
        for sid, sc, y in zip(eval_data.index.tolist(), preds, y_true):
            prs_pred_rows.append({
                'sample_id': sid, 'score': float(sc), 'y_true': int(y),
                'method': method, 'config': config_name, 'eval_set': label,
            })

prs_predictions_df = pd.DataFrame(prs_pred_rows)
out_path = ARTIFACT_DIR / 'prs_predictions.csv'
prs_predictions_df.to_csv(out_path, index=False)
print(f"Saved -> {out_path}  ({len(prs_predictions_df)} rows)")
print(prs_predictions_df.groupby(['method', 'config', 'eval_set']).size().to_string())


## 5. ROC curves — GACRS test + CAMP-only


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for method, color_base in [('PRS-CS', '#1f77b4'), ('PRS-CSx', '#ff7f0e')]:
    ds = data_sources[method]
    for config_name, x_cols, ls in [('PRS only', X_columns_prs, '-'),
                                     ('PRS + PCs', X_columns_full, '--')]:
        model = LogisticRegression(solver='liblinear', max_iter=1000)
        model.fit(ds['train'][x_cols], ds['train']['PHENO'])

        # GACRS test
        te_pred = model.predict_proba(ds['test'][x_cols])[:, 1]
        fpr, tpr, _ = roc_curve(ds['test']['PHENO'], te_pred)
        te_auc = roc_auc_score(ds['test']['PHENO'], te_pred)
        axes[0].plot(fpr, tpr, ls=ls, color=color_base,
                     label=f'{method} {config_name} (AUC={te_auc:.3f})')

        # CAMP-only full
        co_pred = model.predict_proba(ds['camp_only'][x_cols])[:, 1]
        fpr, tpr, _ = roc_curve(ds['camp_only']['PHENO'], co_pred)
        co_auc = roc_auc_score(ds['camp_only']['PHENO'], co_pred)
        axes[1].plot(fpr, tpr, ls=ls, color=color_base,
                     label=f'{method} {config_name} (AUC={co_auc:.3f})')

for ax, title in zip(axes, ['GACRS Test', 'CAMP-only']):
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize=9)

plt.suptitle('PRS-CS vs PRS-CSx (altPRS)', y=1.02)
plt.tight_layout()
plt.show()


## 6. Violin plots by case/control — GACRS test + CAMP-only


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (method, ds) in enumerate(data_sources.items()):
    for col, (name, data) in enumerate([
        ('GACRS Test', ds['test']),
        ('CAMP-only',  ds['camp_only']),
    ]):
        plot_df = pd.DataFrame({
            'PRS':    data['PRS'].values,
            'Status': ['Case' if p == 1 else 'Control' for p in data['PHENO'].values],
        })
        sns.violinplot(data=plot_df, x='Status', y='PRS', order=['Control', 'Case'],
                       palette=['#4393c3', '#d6604d'], cut=0, ax=axes[row, col])

        g1 = data['PRS'][data['PHENO'] == 0].values
        g2 = data['PRS'][data['PHENO'] == 1].values
        _, p = ttest_ind(g1, g2, equal_var=False)
        raw_auc = roc_auc_score(data['PHENO'], data['PRS'])
        axes[row, col].set_title(f'{name}\nAUC={raw_auc:.3f}, P={p:.2e}', fontsize=10)
        axes[row, col].set_ylabel(f'{method} Score' if col == 0 else '')

plt.suptitle('PRS Score Distribution by Case/Control (altPRS)', y=1.00)
plt.tight_layout()
plt.show()
